# Engenharia de Features (v4)
## ELO Médio dos Adversários + Filtro de Jogos Mínimos

**Melhorias em relação às versões anteriores:**

1. **Filtro de jogos mínimos (≥15 jogos no ciclo):** Remove seleções com dados insuficientes para features confiáveis. Seleções com poucos jogos têm médias instáveis que distorcem o modelo.

2. **ELO médio dos adversários como feature:** Em vez de ponderar gols pelo ELO (v2/v3), informamos diretamente ao modelo a força média dos adversários enfrentados. Nova Zelândia jogando contra adversários fracos terá ELO médio baixo — o modelo aprende a descontar suas médias de gols.

```
Features novas:
  elo_medio_adv_ciclo  → ELO médio dos adversários no ciclo completo
  elo_medio_adv_ult15  → ELO médio dos adversários nos últimos 15 jogos
```

**Dataset de saída:** `data/processed/features_completo_v4.csv`

## 1. Imports e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src/features')
from elo import calcular_elo_historico

sns.set_theme(style='whitegrid')

df_raw = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])
print(f'Shape: {df_raw.shape}')

## 2. Calculando ELO Histórico

In [ ]:
df = calcular_elo_historico(df_raw, elo_inicial=1000, k_competitivo=40, k_amistoso=20)
print(f'ELO calculado para {len(df)} jogos')

## 3. Função de Features v4

### Por que ELO médio dos adversários é mais direto que ponderar gols?

Nas versões anteriores (v2/v3) multiplicamos os gols pelo ELO do adversário — isso cria uma feature composta que mistura **volume de gols** com **qualidade do adversário**, dificultando a interpretação do modelo.

Na v4 mantemos as features originais de gols **separadas** e adicionamos o ELO médio como feature independente. Assim o modelo pode aprender:

> *"Esta seleção marcou 2 gols por jogo, MAS seus adversários tinham ELO médio de 900 — portanto, ajustar para baixo"*

vs.

> *"Esta seleção marcou 2 gols por jogo e seus adversários tinham ELO médio de 1400 — previsão alta justificada"*

In [ ]:
def calcular_features_v4(selecao, ciclo, copa, min_jogos=15):
    """
    Calcula features v4:
    - Features v1 (7 features originais)
    - ELO médio dos adversários no ciclo e nos últimos 15 jogos
    - Filtro de jogos mínimos
    """
    jogos = ciclo[
        (ciclo['home_team'] == selecao) |
        (ciclo['away_team'] == selecao)
    ].sort_values('date')

    # Filtro de jogos mínimos
    if len(jogos) < min_jogos:
        raise ValueError(f'Apenas {len(jogos)} jogos — mínimo exigido: {min_jogos}')

    gm, gs, vit, elo_adv = [], [], [], []

    for _, row in jogos.iterrows():
        if row['home_team'] == selecao:
            gm.append(row['home_score'])
            gs.append(row['away_score'])
            vit.append(1 if row['home_score'] > row['away_score'] else 0)
            elo_adv.append(row['elo_away_antes'])
        else:
            gm.append(row['away_score'])
            gs.append(row['home_score'])
            vit.append(1 if row['away_score'] > row['home_score'] else 0)
            elo_adv.append(row['elo_home_antes'])

    gm      = np.array(gm)
    gs      = np.array(gs)
    vit     = np.array(vit)
    elo_adv = np.array(elo_adv)

    # Últimos 15 jogos
    ult15 = jogos.tail(15)
    gm15, gs15, vit15, elo_adv15 = [], [], [], []

    for _, row in ult15.iterrows():
        if row['home_team'] == selecao:
            gm15.append(row['home_score'])
            gs15.append(row['away_score'])
            vit15.append(1 if row['home_score'] > row['away_score'] else 0)
            elo_adv15.append(row['elo_away_antes'])
        else:
            gm15.append(row['away_score'])
            gs15.append(row['home_score'])
            vit15.append(1 if row['away_score'] > row['home_score'] else 0)
            elo_adv15.append(row['elo_home_antes'])

    gm15      = np.array(gm15)
    gs15      = np.array(gs15)
    vit15     = np.array(vit15)
    elo_adv15 = np.array(elo_adv15)

    # Target
    selecao_copa = copa[
        (copa['home_team'] == selecao) |
        (copa['away_team'] == selecao)
    ]
    gols_copa = [
        row['home_score'] if row['home_team'] == selecao else row['away_score']
        for _, row in selecao_copa.iterrows()
    ]

    return {
        # Features v1
        'media_gols_marcados_ciclo': gm.mean(),
        'media_gols_sofridos_ciclo': gs.mean(),
        'pct_vitorias_ciclo':        vit.mean(),
        'total_jogos_ciclo':         len(jogos),
        'media_gols_marcados_ult15': gm15.mean(),
        'media_gols_sofridos_ult15': gs15.mean(),
        'pct_vitorias_ult15':        vit15.mean(),
        # Features v4 — ELO médio dos adversários
        'elo_medio_adv_ciclo':       elo_adv.mean(),
        'elo_medio_adv_ult15':       elo_adv15.mean(),
        # Target
        'media_gols_copa':           np.mean(gols_copa)
    }

print('Função v4 definida!')

### Teste com o Brasil — Copa 2022

In [ ]:
copa_2022 = df[
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'].dt.year == 2022)
]
ciclo_2022 = df[
    (df['date'] >= '2018-07-16') &
    (df['date'] <= '2022-11-19') &
    (df['tournament'] != 'FIFA World Cup')
]

brasil = calcular_features_v4('Brazil', ciclo_2022, copa_2022)
print('Features v4 — Brasil (Copa 2022):')
for k, v in brasil.items():
    print(f'  {k:<35} {v:.4f}')

## 4. Pipeline Completo — Todas as Copas (1994–2022)

In [ ]:
copas = {
    1994: {'ciclo_inicio': '1990-07-09', 'ciclo_fim': '1994-06-16', 'copa_inicio': '1994-06-17', 'copa_fim': '1994-07-17'},
    1998: {'ciclo_inicio': '1994-07-18', 'ciclo_fim': '1998-06-09', 'copa_inicio': '1998-06-10', 'copa_fim': '1998-07-12'},
    2002: {'ciclo_inicio': '1998-07-13', 'ciclo_fim': '2002-05-30', 'copa_inicio': '2002-05-31', 'copa_fim': '2002-06-30'},
    2006: {'ciclo_inicio': '2002-07-01', 'ciclo_fim': '2006-06-08', 'copa_inicio': '2006-06-09', 'copa_fim': '2006-07-09'},
    2010: {'ciclo_inicio': '2006-07-10', 'ciclo_fim': '2010-06-10', 'copa_inicio': '2010-06-11', 'copa_fim': '2010-07-11'},
    2014: {'ciclo_inicio': '2010-07-12', 'ciclo_fim': '2014-06-11', 'copa_inicio': '2014-06-12', 'copa_fim': '2014-07-13'},
    2018: {'ciclo_inicio': '2014-07-14', 'ciclo_fim': '2018-06-13', 'copa_inicio': '2018-06-14', 'copa_fim': '2018-07-15'},
    2022: {'ciclo_inicio': '2018-07-16', 'ciclo_fim': '2022-11-19', 'copa_inicio': '2022-11-20', 'copa_fim': '2022-12-18'},
}

todos_dados = []
filtrados   = []

for ano, datas in copas.items():
    print(f'Processando Copa {ano}...')

    ciclo = df[
        (df['date'] >= datas['ciclo_inicio']) &
        (df['date'] <= datas['ciclo_fim']) &
        (df['tournament'] != 'FIFA World Cup')
    ]
    copa = df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= datas['copa_inicio']) &
        (df['date'] <= datas['copa_fim'])
    ]

    selecoes = pd.unique(copa[['home_team', 'away_team']].values.ravel())

    for selecao in selecoes:
        try:
            resultado = calcular_features_v4(selecao, ciclo, copa, min_jogos=15)
            resultado['selecao']   = selecao
            resultado['copa_alvo'] = ano
            todos_dados.append(resultado)
        except ValueError as e:
            filtrados.append({'selecao': selecao, 'copa': ano, 'motivo': str(e)})
        except Exception as e:
            print(f'  Erro inesperado em {selecao}: {e}')

df_final = pd.DataFrame(todos_dados)
df_filtrados = pd.DataFrame(filtrados)

print(f'\nDataset final: {df_final.shape[0]} linhas × {df_final.shape[1]} colunas')
print(f'Seleções filtradas (< 15 jogos): {len(df_filtrados)}')
if len(df_filtrados) > 0:
    print(df_filtrados.to_string(index=False))

## 5. Validação — ELO Médio dos Adversários

Verificamos se o ELO médio dos adversários faz sentido — seleções como Brasil e Alemanha devem ter ELO médio de adversários maior que seleções menores.

In [ ]:
# ELO médio por seleção em 2022
df_2022 = df_final[df_final['copa_alvo'] == 2022][['selecao', 'media_gols_marcados_ciclo', 'elo_medio_adv_ciclo']]
df_2022 = df_2022.sort_values('elo_medio_adv_ciclo', ascending=False)

print('ELO médio dos adversários — Copa 2022 (top 10 e bottom 5):')
print(pd.concat([df_2022.head(10), df_2022.tail(5)]).to_string(index=False))

# Correlação entre ELO médio dos adversários e gols na Copa
corr = df_final['elo_medio_adv_ciclo'].corr(df_final['media_gols_copa'])
print(f'\nCorrelação ELO médio adversários vs gols na Copa: {corr:.4f}')

In [ ]:
# Scatter: ELO médio adversários vs gols na Copa
plt.figure(figsize=(10, 5))
plt.scatter(df_final['elo_medio_adv_ciclo'], df_final['media_gols_copa'],
            alpha=0.5, color='steelblue', edgecolors='black', s=40)

z = np.polyfit(df_final['elo_medio_adv_ciclo'], df_final['media_gols_copa'], 1)
x_line = np.linspace(df_final['elo_medio_adv_ciclo'].min(),
                     df_final['elo_medio_adv_ciclo'].max(), 100)
plt.plot(x_line, np.poly1d(z)(x_line), 'r--', alpha=0.7, label=f'Correlação: {corr:.2f}')

plt.xlabel('ELO Médio dos Adversários no Ciclo')
plt.ylabel('Média de Gols na Copa (target)')
plt.title('Força dos Adversários no Ciclo vs. Desempenho na Copa\n(todas as Copas 1994–2022)')
plt.legend()
plt.tight_layout()
plt.savefig('../article/figures/elo_adv_vs_gols_copa.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Salvamento

In [ ]:
print('Linhas por Copa:')
print(df_final['copa_alvo'].value_counts().sort_index())

df_final.to_csv('../data/processed/features_completo_v4.csv', index=False)
print('\nDataset v4 salvo em: data/processed/features_completo_v4.csv')

## 7. Resumo das Features v4

| Feature | Versão | Descrição |
|---------|--------|-----------|
| `media_gols_marcados_ciclo` | v1 | Média de gols marcados no ciclo |
| `media_gols_sofridos_ciclo` | v1 | Média de gols sofridos no ciclo |
| `pct_vitorias_ciclo` | v1 | % de vitórias no ciclo |
| `total_jogos_ciclo` | v1 | Total de jogos no ciclo |
| `media_gols_marcados_ult15` | v1 | Média de gols nos últimos 15 jogos |
| `media_gols_sofridos_ult15` | v1 | Média de gols sofridos nos últimos 15 |
| `pct_vitorias_ult15` | v1 | % de vitórias nos últimos 15 jogos |
| `elo_medio_adv_ciclo` | **v4** | ELO médio dos adversários no ciclo |
| `elo_medio_adv_ult15` | **v4** | ELO médio dos adversários nos últimos 15 |

**Filtro aplicado:** Seleções com menos de 15 jogos no ciclo são excluídas do dataset.

**Próximo passo:** `03_modelos_v4.ipynb` — avaliar se a v4 melhora o MAE.